# 5) EE-2.3 The AC power flow — Exercises

**Goals:** size a power flow problem, build a Y-bus from a line list, write a general Newton power flow for PQ and PV buses, check it against pandapower on a meshed network, find a network's loadability margin, and handle a generator's reactive power limit.

Units: kV, Ω, S, MW, Mvar (V · (Y·V)\* is then in MVA). Injections are positive into the bus. Bus 0 is the slack at 220 kV, angle 0. The helper `pandapower_3bus` builds the same network in pandapower for checking.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [ ]:
import math
import warnings

import numpy as np
import pandas as pd
import pandapower as pp

warnings.filterwarnings("ignore", category=FutureWarning)

V_NOM = 220.0                                          # kV
LINES = [(0, 1, 3 + 30j), (1, 2, 4 + 40j), (0, 2, 5 + 50j)]   # (from, to, Z in ohm): a meshed 3-bus network
LOADS = {1: 100 + 30j, 2: 120 + 40j}                   # MVA consumed


def pandapower_3bus(gen_at_2=False, q_max=None, load_scale=1.0):
    """The same network in pandapower (bus 0 = slack at 1.0 pu), for checking your results."""
    net = pp.create_empty_network()
    b = [pp.create_bus(net, vn_kv=V_NOM) for _ in range(3)]
    pp.create_ext_grid(net, b[0], vm_pu=1.0)
    for f, t, z in LINES:
        pp.create_line_from_parameters(net, b[f], b[t], length_km=1, r_ohm_per_km=z.real,
                                       x_ohm_per_km=z.imag, c_nf_per_km=0.0, max_i_ka=3.0)
    for bus, s in LOADS.items():
        pp.create_load(net, b[bus], p_mw=s.real * load_scale, q_mvar=s.imag * load_scale)
    if gen_at_2:
        pp.create_gen(net, b[2], p_mw=0.0, vm_pu=1.0,
                      max_q_mvar=1e6 if q_max is None else q_max, min_q_mvar=-1e6)
    pp.runpp(net, numba=False, enforce_q_lims=q_max is not None)
    return net


def numerical_jacobian(F, x, h=1e-6):
    """Central differences (M1.3)."""
    return np.column_stack([(F(x + e) - F(x - e)) / (2 * h) for e in np.eye(len(x)) * h])

### Warm-ups

**1. How big is a power flow?**

Return `(unknowns, equations)` for a network with the given numbers of slack, PV and PQ buses.

In [ ]:
def power_flow_size(n_slack: int, n_pv: int, n_pq: int) -> tuple[int, int]:
    """Number of unknowns and of equations in the polar power flow formulation."""
    ...  # YOUR CODE HERE

In [ ]:
assert power_flow_size(1, 5, 20) == (45, 45)
assert power_flow_size(1, 0, 1) == (2, 2), "the lecture's two-bus case"
assert power_flow_size(1, 1, 0) == (1, 1), "the lecture's PV-bus case"

**2. A Y-bus from a line list**

Write `build_ybus(n_bus, lines)` for lines `(from, to, Z)`. Unlike EE-1, there is no separate reference node: every bus is a row. Check symmetry, and that every row sums to zero (no shunt elements yet).

In [ ]:
def build_ybus(n_bus: int, lines: list[tuple[int, int, complex]]) -> np.ndarray:
    """Complex bus admittance matrix (S)."""
    ...  # YOUR CODE HERE

In [ ]:
Y3 = build_ybus(3, LINES)
assert np.allclose(Y3, Y3.T), "symmetric"
assert np.allclose(Y3.sum(axis=1), 0), "rows sum to zero without shunts"
assert np.isclose(Y3[0, 1], -1 / (3 + 30j))
Y2 = build_ybus(2, [(0, 1, 5 + 40j)])
assert np.allclose(Y2, np.array([[1, -1], [-1, 1]]) / (5 + 40j)), "the lecture's 2x2 Y-bus"

### Core

**3. A general Newton power flow**

Write `power_flow(Y, S_inj, pv=None)`:
- bus 0 is the slack (220 kV, angle 0);
- `S_inj` holds the specified complex injection of every bus (MVA; the slack entry is ignored, and only the real part is used at PV buses);
- `pv` maps PV buses to their voltage magnitude (kV);
- unknowns: the angle of every non-slack bus and the magnitude of every PQ bus, from a flat start;
- equations: ΔP at every non-slack bus, ΔQ at every PQ bus.

Return the voltage phasors (kV) and the number of iterations; raise `RuntimeError` if it does not converge.

In [ ]:
def power_flow(Y: np.ndarray, S_inj, pv: dict | None = None, tol: float = 1e-8, max_iter: int = 30):
    """Polar Newton power flow. Returns (V phasors in kV, iterations)."""
    ...  # YOUR CODE HERE

In [ ]:
S_inj = np.array([0, -LOADS[1], -LOADS[2]])
V, iters = power_flow(Y3, S_inj)
net = pandapower_3bus()
assert np.allclose(np.abs(V) / V_NOM, net.res_bus.vm_pu, atol=1e-8), "magnitudes must match pandapower"
assert np.allclose(np.degrees(np.angle(V)), net.res_bus.va_degree, atol=1e-7), "angles must match pandapower"
S = V * np.conj(Y3 @ V)
assert np.isclose(S[0].real, net.res_ext_grid.p_mw[0]), "slack injection matches pandapower's ext_grid supply"
assert np.allclose(S[1:], S_inj[1:]), "loads are met exactly"
print(f"{iters} iterations; |V| = {np.round(np.abs(V) / V_NOM, 6)} pu, angles = {np.round(np.degrees(np.angle(V)), 4)} deg")

**4. Interpretation**

Why does a power flow start from a flat start (all magnitudes at nominal, all angles zero)? What would you check first if Newton fails to converge on a real network — and why is "the power flow does not converge" sometimes a correct answer rather than a bug?

*Your answer here.*

### Challenge

**5. How much more load can this network take?**

Scale **both** loads by a factor λ (same power factors). Write `loadability_margin(Y, lam_hi)`: find the largest λ for which `power_flow` converges, to a precision of 0.01, by bisection. Then report the voltages just before the limit.

In [ ]:
def loadability_margin(Y: np.ndarray, lam_hi: float = 20.0, precision: float = 0.01) -> float:
    """Largest load scaling factor with a converging power flow (bisection)."""
    ...  # YOUR CODE HERE

In [ ]:
lam = loadability_margin(Y3)
V_lim, _ = power_flow(Y3, np.array([0, -LOADS[1], -LOADS[2]]) * lam)
assert 1.0 < lam < 20.0
try:
    power_flow(Y3, np.array([0, -LOADS[1], -LOADS[2]]) * (lam + 0.05))
    raise AssertionError("slightly beyond the margin there should be no solution")
except (RuntimeError, np.linalg.LinAlgError):
    pass
assert (np.abs(V_lim[1:]) / V_NOM < 0.8).all(), "near the limit the voltages have collapsed far below nominal"
print(f"loads can grow by a factor {lam:.2f} (to {lam * 220:.0f} MW in total); "
      f"voltages just before the limit: {np.round(np.abs(V_lim) / V_NOM, 3)} pu")

**6. A generator with a reactive power limit**

A generator at bus 2 holds 220 kV (a PV bus) and produces no active power — but it can supply at most `q_max` Mvar. Write `pv_with_q_limit(Y, q_max)`:
1. solve with bus 2 as a PV bus and compute the reactive power the generator must supply;
2. if that exceeds `q_max`, fix the generator at `q_max`, turn bus 2 into a PQ bus and solve again.

Return the voltages, the generator's Q and whether the limit is active. Check against pandapower with and without the limit.

In [ ]:
def pv_with_q_limit(Y: np.ndarray, q_max: float) -> tuple[np.ndarray, float, bool]:
    """Voltages (kV), generator Q (Mvar) and whether the Q limit binds, for a voltage-controlling generator at bus 2."""
    ...  # YOUR CODE HERE

In [ ]:
V_free, q_free, limited = pv_with_q_limit(Y3, q_max=1e6)
ref = pandapower_3bus(gen_at_2=True)
assert not limited and np.isclose(q_free, ref.res_gen.q_mvar[0], rtol=1e-6)
assert np.isclose(abs(V_free[2]), V_NOM), "voltage held at 220 kV"

V_lim, q_lim, limited = pv_with_q_limit(Y3, q_max=40.0)
ref_lim = pandapower_3bus(gen_at_2=True, q_max=40.0)
assert limited and q_lim == 40.0
assert np.allclose(np.abs(V_lim) / V_NOM, ref_lim.res_bus.vm_pu, atol=1e-7), "must match pandapower with Q limits"
assert abs(V_lim[2]) < V_NOM, "at its limit the generator can no longer hold the voltage"
print(f"unlimited: Q = {q_free:.2f} Mvar, V2 = {abs(V_free[2]) / V_NOM:.4f} pu | "
      f"q_max = 40: Q = {q_lim:.0f} Mvar, V2 = {abs(V_lim[2]) / V_NOM:.4f} pu")

**7. Interpretation**

A grid operator sees the generator at bus 2 reach its reactive power limit during a demand peak. What happens to the voltages, why is this a dangerous moment for voltage stability (connect it to task 5), and which actions could the operator take?

*Your answer here.*